[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-05.ipynb)

# 5주차 실습: 평가 시스템과 지표가 측정하지 못하는 것

**목표.** 4주차의 제주 관광 문의 채점셋 24건과 같은 모델을 그대로 써서 세 가지를 해 본다.

1. **규칙 채점**(정답지 비교)으로 정확도, 혼동행렬, 부류별 F1을 구한다
2. **judge 채점**(정답지 없이 문서만 준 판정)과 대조해 불일치와 오답 승인(false acceptance) 사례를 모은다
3. **pass@k 곡선**을 그려, 프롬프트 개선(예시 2개 추가)이 새 능력인지 탐색 효율 개선인지 판정한다

강의 노트 5주차와 같은 흐름이다. judge도 다음 토큰 확률 기계라는 점에 주목하자.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers torch

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 채점셋 - 4주차의 제주 관광 문의 24건을 그대로

4주차와 같은 채점셋이다. 라벨은 주차, 시설, 요금 세 가지다.

In [ ]:
# 제주 관광 문의 채점셋 (4주차와 동일)
eval_set = [
    ("성산일출봉 주차장이 어디예요?", "주차"),
    ("함덕해수욕장에 주차할 곳이 있나요?", "주차"),
    ("제주공항에서 렌터카를 어디서 반납하나요?", "주차"),
    ("만장굴 주차 요금이 있나요?", "주차"),
    ("협재해수욕장 주차장이 만차인지 알 수 있나요?", "주차"),
    ("한라산 어리목 탐방로 주차가 가능한가요?", "주차"),
    ("천지연폭포 주차장에서 입구까지 멀어요?", "주차"),
    ("섭지코지 주차 공간이 넓은가요?", "주차"),
    ("이호테우해수욕장에 샤워실이 있나요?", "시설"),
    ("성산일출봉에 화장실이 많이 있나요?", "시설"),
    ("함덕해수욕장에 파라솔을 빌릴 수 있나요?", "시설"),
    ("제주민속촌에 수유실이 있나요?", "시설"),
    ("한라산 국립공원에 매점이 있나요?", "시설"),
    ("월정리해수욕장에 짐 보관함이 있나요?", "시설"),
    ("만장굴 안을 휠체어가 다닐 수 있나요?", "시설"),
    ("식물원에 유모차 대여가 되나요?", "시설"),
    ("성산일출봉 입장료가 얼마예요?", "요금"),
    ("만장굴 입장료 할인이 있나요?", "요금"),
    ("제주민속촌 가족권 가격이 어떻게 되나요?", "요금"),
    ("식물원 입장권을 온라인으로 사면 더 싼가요?", "요금"),
    ("우도 왕복 배 삯이 얼마인가요?", "요금"),
    ("한라산 트레킹은 무료인가요?", "요금"),
    ("청소년은 입장료가 할인되나요?", "요금"),
    ("오름 이용 요금이 따로 있나요?", "요금"),
]

LABELS = ["주차", "시설", "요금"]
print(f"채점셋: {len(eval_set)}건")
for lab in LABELS:
    print(f"  {lab}: {sum(1 for _, y in eval_set if y == lab)}건")

### 1-2. 모델 로딩 - 가중치는 그대로

4주차와 같은 작은 다국어 언어모형(Qwen3-0.6B-Base)을 쓴다. 학습시키지 않는다. 내려받은 가중치 그대로다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

total = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터: {total:,}")
print("이번 주에 갱신하는 파라미터: 0 (모델은 그대로, 채점 방식만 비교한다)")

### 1-3. 라벨 확률로 분류하기 - 4주차의 방식 그대로

프롬프트 바로 다음 토큰 자리의 확률분포에서 각 라벨 첫 토큰의 확률을 비교해 분류한다. 4주차와 같은 방식이다.

프롬프트는 4주차의 두 개를 가져온다. **A(지시문만)** 는 규칙 채점과 judge 채점 대조에 쓰고, **A와 B(예시 2개)** 는 뒤의 pass@k 대조에 쓴다.

In [ ]:
import torch

def label_scores(prompt_text):
    # 프롬프트 바로 다음 토큰 자리에서 각 라벨 첫 토큰의 로그확률을 비교한다
    prefix_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(prefix_ids).logits
    next_token_logps = torch.log_softmax(logits[0, -1], dim=-1)
    return {lab: next_token_logps[tokenizer(" " + lab, add_special_tokens=False).input_ids[0]].item()
            for lab in LABELS}

PROMPT_A = """아래 제주 관광 문의를 주차, 시설, 요금 중 하나로 분류하시오.
라벨만 답하시오.

문의: {query}
라벨:"""

PROMPT_B = """아래 제주 관광 문의를 주차, 시설, 요금 중 하나로 분류하시오.
라벨만 답하시오.

문의: 함덕해수욕장 주차장이 어디예요?
라벨: 주차

문의: 오름 입장료 할인이 있나요?
라벨: 요금

문의: {query}
라벨:"""

# 동작 확인: 4주차와 같은 문의로 한 건 맞혀 보기
trial = PROMPT_A.format(query="섭지코지 주차 공간이 넓은가요?")
trial_scores = label_scores(trial)
for lab, sc in trial_scores.items():
    print(f"  {lab}: {sc:.3f}")
print("-> 모델의 선택:", max(trial_scores, key=trial_scores.get))

### 1-4. 규칙 채점 - 정확도, 혼동행렬, 부류별 F1

프롬프트 A로 채점셋 전체를 돌려 정답지와 비교한다. **정답지와 비교하므로 틀린 것은 틀렸다고 나온다.** judge 채점과 대조할 기준선이다.

정확도 하나로 끝내지 않고 **혼동행렬**(행: 정답, 열: 예측)과 **부류별 정밀도·재현율·F1**까지 출력한다. 강의 노트의 "실측 예" 표가 여기서 나온다. 정확도가 가리는 치우침이 어느 부류에서 어느 부류로 새는지 확인하자.

In [ ]:
def rule_evaluate(prompt):
    results = []
    for sentence, gold in eval_set:
        scores = label_scores(prompt.format(query=sentence))
        pred = max(scores, key=scores.get)
        results.append((sentence, gold, pred))
    return results

rule_results = rule_evaluate(PROMPT_A)
rule_correct = sum(1 for _, gold, pred in rule_results if gold == pred)
print(f"규칙 채점 정확도: {rule_correct}/{len(eval_set)} = {rule_correct/len(eval_set):.2f}")

# 혼동행렬: 행 = 정답, 열 = 예측
print("\n혼동행렬 (행: 정답, 열: 예측)")
print("      " + "".join(f"{lab:>6}" for lab in LABELS))
for g in LABELS:
    row = [sum(1 for _, gold, pred in rule_results if gold == g and pred == p) for p in LABELS]
    print(f"{g:>4}  " + "".join(f"{n:>7}" for n in row))

# 부류별 정밀도, 재현율, F1
print(f"\n{'부류':<4} {'TP':>3} {'FP':>3} {'FN':>3} {'정밀도':>6} {'재현율':>6} {'F1':>6}")
f1s = []
for lab in LABELS:
    tp = sum(1 for _, g, p in rule_results if g == lab and p == lab)
    fp = sum(1 for _, g, p in rule_results if g != lab and p == lab)
    fn = sum(1 for _, g, p in rule_results if g == lab and p != lab)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    f1s.append(f1)
    print(f"{lab:<4} {tp:>3} {fp:>3} {fn:>3} {precision:>8.2f} {recall:>8.2f} {f1:>6.2f}")
print(f"macro-F1 = {sum(f1s) / len(f1s):.2f}")

print("\n틀린 문항:")
for sentence, gold, pred in rule_results:
    if gold != pred:
        print(f"  {sentence}  (정답: {gold} / 모델: {pred})")

### 1-5. judge 채점 - 문서만 준 판정 프롬프트

이제 judge 채점이다. 모델에게 **루브릭(채점 기준표) 문서만** 주고, 1-4의 모델 출력이 맞는지 판정하게 한다. **정답지는 주지 않는다.** 강의 노트의 false acceptance 실험과 같은 설정이다.

judge 판정도 다음 토큰 확률 비교다. "정답"과 "오답"의 첫 토큰 중 어느 쪽에 확률을 더 두는지 본다. 두 확률만 놓고 합이 1이 되게 다시 나눈 값을 **P(정답)** 이라 부르자. P(정답)이 0.5보다 크면 판정은 "정답"이다.

> 판정 단어를 고를 때도 토큰을 확인해야 한다. 이 모델의 토크나이저는 "틀림"의 첫 토큰을 온전한 글자가 아니라 **한글 글자의 앞부분 바이트 조각**으로 자른다. 그 조각은 여러 한글 글자가 공유하므로 "틀림"의 확률로 읽을 수 없다. "정답"(' 정' + '답')과 "오답"(' 오' + '답')은 첫 토큰이 온전한 글자라 "틀림"보다 낫다. 그래도 ' 정'으로 시작하는 다른 단어(정보, 정말 등)의 확률이 섞이는 한계는 남는다. 두 첫 토큰만 놓고 다시 나누므로 "어느 쪽으로 기우는가"를 비교하는 데는 쓸 수 있다.

In [ ]:
# judge 판정 프롬프트. 정답지는 넣지 않는다. 문서(루브릭)만 준다
JUDGE_PROMPT = """[역할] 당신은 분류 채점자입니다.
아래의 문의와 모델의 분류 출력을 보고, 출력이 문의를 정확히 분류했는지 판정하시오.
답은 "정답" 또는 "오답"으로만 쓰시오.

[루브릭]
- 출력의 라벨이 문의의 핵심 요구에 부합하면 "정답"
- 출력의 라벨이 문의의 핵심 요구와 어긋나면 "오답"

[문의] {query}
[모델 출력] 라벨: {pred}

[판정]"""

VERDICTS = ["정답", "오답"]

def judge_prob(prompt_text):
    # "정답"과 "오답" 첫 토큰의 확률만 놓고 다시 나눈 P(정답)을 돌려준다
    prefix_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(prefix_ids).logits
    next_token_logps = torch.log_softmax(logits[0, -1], dim=-1)
    logps = torch.tensor([next_token_logps[tokenizer(" " + v, add_special_tokens=False).input_ids[0]].item()
                          for v in VERDICTS])
    return torch.softmax(logps, dim=0)[0].item()

def judge_verdict(p_correct):
    return "정답" if p_correct > 0.5 else "오답"

# 동작 확인: 규칙 채점의 첫 문항을 judge에게도 물어 본다
trial_sentence, trial_gold, trial_pred = rule_results[0]
p = judge_prob(JUDGE_PROMPT.format(query=trial_sentence, pred=trial_pred))
print(f"문의: {trial_sentence}")
print(f"정답: {trial_gold} / 모델: {trial_pred}")
print(f"judge: P(정답) = {p:.2f} -> 판정 {judge_verdict(p)}")

### 1-6. 규칙 채점 vs judge 채점 대조 - 불일치 기록

채점셋 전체를 두 방식으로 채점해 나란히 놓는다. judge는 **모델 출력이 문의에 부합하는가**만 보고 판정하며, 정답지를 모른다.

judge가 "정답"이라는데 정답지 기준으로는 틀린 문항이 바로 **false acceptance** 사례다.

In [ ]:
judge_nogold = [judge_prob(JUDGE_PROMPT.format(query=s, pred=p)) for s, g, p in rule_results]

print(f"{'규칙':<4} {'judge':<5} {'P(정답)':>7}  문의 (정답 -> 모델)")
print("-" * 72)
mismatch_count = 0
false_accept = []
for (sentence, gold, pred), pc in zip(rule_results, judge_nogold):
    rule_ok = gold == pred
    judge_ok = judge_verdict(pc) == "정답"
    flag = "  <-- 불일치" if rule_ok != judge_ok else ""
    if rule_ok != judge_ok:
        mismatch_count += 1
    if judge_ok and not rule_ok:
        false_accept.append((sentence, gold, pred, pc))
    print(f"{'정답' if rule_ok else '오답':<4} {judge_verdict(pc):<5} {pc:>7.2f}  {sentence} ({gold} -> {pred}){flag}")

print()
print(f"규칙 채점과 judge 채점의 불일치: {mismatch_count}건 / {len(eval_set)}건")
print(f"그중 judge가 오답을 승인한 사례(false acceptance): {len(false_accept)}건")
print()
print("** 수집된 오답 승인 사례 (학습활동에 그대로 쓴다) **")
for sentence, gold, pred, pc in false_accept:
    print(f"  문의: {sentence}")
    print(f"  정답: {gold} / 모델 출력: {pred} / judge: 정답 (P(정답) = {pc:.2f})")
    print("  -> judge는 정답지를 모른다. 이 '정답'은 정답지와 대조해 확인한 판정이 아니다")
    print()

### 1-7. 정답지를 넣으면? - 판정 단어 대신 확률을 본다

같은 judge 프롬프트에 **정답지 한 줄**(`[정답지] 라벨: ...`)만 더해 다시 채점한다. 강의 노트의 judge 실험(정답지 유무 토글)과 같은 설정이다.

두 가지를 본다.

- 판정 단어("정답" 개수)가 바뀌는가
- **맞은 문항과 틀린 문항의 P(정답) 범위**가 어떻게 달라지는가. 두 범위가 겹치지 않으면 어떤 기준선(임계값)으로 깔끔하게 가를 수 있다는 뜻이다. 겹치면 어떤 기준선도 둘을 가르지 못한다

In [ ]:
JUDGE_PROMPT_GOLD = JUDGE_PROMPT.replace("[판정]", "[정답지] 라벨: {gold}\n\n[판정]")
judge_gold = [judge_prob(JUDGE_PROMPT_GOLD.format(query=s, pred=p, gold=g)) for s, g, p in rule_results]

def p_range(probs, want_correct):
    vals = [pc for (s, g, p), pc in zip(rule_results, probs) if (g == p) == want_correct]
    return f"{min(vals):.2f} ~ {max(vals):.2f}" if vals else "-"

print(f"{'조건':<8} {'판정이 정답':>10}   {'맞은 문항 P(정답)':<16} {'틀린 문항 P(정답)':<16}")
for name, probs in [("정답지 없음", judge_nogold), ("정답지 포함", judge_gold)]:
    n_yes = sum(judge_verdict(pc) == "정답" for pc in probs)
    print(f"{name:<8} {n_yes:>7}/{len(probs)}   {p_range(probs, True):<20} {p_range(probs, False):<20}")

### 1-8. pass@k - k번의 시도 안에 맞히는 비율

이제 **pass@k**를 측정한다. 한 문항에 대해 모델이 다음 토큰을 n번 뽑게 하고(샘플링), 정답 라벨의 첫 토큰이 몇 번(c) 나왔는지 센다. 그다음 강의 노트의 무편 추정량으로 pass@k를 계산한다.

pass@k = 1 - C(n-c, k) / C(n, k)

세 가지를 알아 두자.

- **라벨 3개 안에서 뽑지 않는다.** 3개 중에서만 뽑으면 아무렇게나 찍어도 pass@3이 0.70이다. 그래서 모델의 **전체 어휘**에서 다음 토큰을 뽑는다. 모델이 엉뚱한 토큰을 내면 그 시도는 실패다
- **단순화가 하나 있다.** 첫 토큰만 보므로 ' 주'로 시작하는 다른 단어(예: '주소')가 나와도 정답으로 센다
- **1-4의 정확도와 숫자가 다르다.** 1-4는 라벨 3개의 확률만 비교해 가장 큰 것을 골랐다(A 정확도 0.83). 여기서는 전체 어휘에서 뽑으므로, 지시문만 있는 A는 라벨이 아닌 토큰(문의 속 단어, 공백 등)을 자주 낸다. 그래서 A의 pass@1은 훨씬 낮게 나온다. 예시가 붙은 B는 "라벨: 주차" 같은 답의 형식을 보여 주므로 확률이 라벨 토큰으로 모인다

비교할 두 프롬프트는 4주차의 A(지시문만)와 B(예시 2개)다. 가중치가 같으므로 B가 모델에 새 능력을 만들 수는 없다. 이 사실을 pass@k 곡선이 어떻게 보여 주는지 확인하는 **대조 실험**이다.

문항당 n = 200번 뽑는다(원전과 같은 값). 모델은 문항마다 한 번만 돌려 확률분포를 얻고, 그 분포에서 200번 뽑는 것은 계산이 거의 들지 않는다. GPU 없이 금방 끝난다.

In [ ]:
import math

torch.manual_seed(42)  # 재현성을 위한 시드. 바꾸면 값이 조금 흔들린다
N_SAMPLES = 200  # 문항당 시행 횟수 n (원전과 같은 값)

def pass_at_k(c, n, k):
    # 무편 추정량: n개 중 k개를 골랐을 때 모두 오답일 확률을 1에서 뺀다
    if not 1 <= k <= n:
        raise ValueError(f"k={k}는 1 이상, 문항당 뽑은 횟수 n={n} 이하여야 한다")
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def sample_counts(prompt, n=N_SAMPLES):
    # 문항마다 다음 토큰을 전체 어휘에서 n번 뽑아, 정답 라벨 첫 토큰이 나온 횟수 c를 센다
    counts = []
    for sentence, gold in eval_set:
        prefix_ids = tokenizer(prompt.format(query=sentence), return_tensors="pt").input_ids
        with torch.no_grad():
            probs = torch.softmax(model(prefix_ids).logits[0, -1], dim=-1)
        gold_id = tokenizer(" " + gold, add_special_tokens=False).input_ids[0]
        draws = torch.multinomial(probs, n, replacement=True)
        counts.append(int((draws == gold_id).sum()))
    return counts

def pass_at_k_curve(counts, k_list, n=N_SAMPLES):
    # 채점셋 전체의 pass@k = 문항별 pass@k의 평균
    return {k: sum(pass_at_k(c, n, k) for c in counts) / len(counts) for k in k_list}

counts_A = sample_counts(PROMPT_A)
counts_B = sample_counts(PROMPT_B)
print(f"문항별 정답 횟수 c ({N_SAMPLES}번 중)")
print("  A:", counts_A)
print("  B:", counts_B)

K_LIST = [1, 2, 4, 8]
curve_A = pass_at_k_curve(counts_A, K_LIST)
curve_B = pass_at_k_curve(counts_B, K_LIST)
print(f"\n{'k':<4} {'A (지시문만)':>12} {'B (예시 2개)':>12}")
for k in K_LIST:
    print(f"{k:<4} {curve_A[k]:>14.3f} {curve_B[k]:>14.3f}")

## 2. 한 지점만 바꿔 보기 - k 목록을 늘려 곡선을 끝까지 그린다

아래 셀의 `# TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 규칙: `K_LIST`를 n(=200)까지 늘려 곡선을 끝까지 그린다. 뽑기는 다시 하지 않고 1-8에서 센 c로 계산만 다시 하므로 금방 끝난다. k는 200을 넘을 수 없다. 늘린 뒤 두 가지를 확인한다. (1) A 곡선이 몇 번째 k에서 B의 pass@1 높이에 닿는가. (2) k = 20에서 멈췄다면 A와 B의 끝값을 어떻게 읽었을까. 끝까지 그리면 무엇이 달라지는가.

In [ ]:
# TODO: K_LIST를 200까지 늘린다. 예: [1, 2, 4, 8, 16, 20, 32, 64, 128, 200]
K_LIST = [1, 2, 4, 8]

# 아래는 그대로 둡니다
K_LIST = sorted(set(K_LIST))  # 순서가 섞이거나 겹쳐도 곡선이 바르게 그려지도록
curve_A = pass_at_k_curve(counts_A, K_LIST)
curve_B = pass_at_k_curve(counts_B, K_LIST)
print(f"{'k':<4} {'A (지시문만)':>12} {'B (예시 2개)':>12}")
for k in K_LIST:
    print(f"{k:<4} {curve_A[k]:>14.3f} {curve_B[k]:>14.3f}")

b1 = pass_at_k_curve(counts_B, [1])[1]
reach = next((k for k in K_LIST if curve_A[k] >= b1), None)
print(f"\nB의 pass@1 = {b1:.3f}")
if reach:
    print(f"A 곡선은 k={reach}에서 처음 이 높이에 닿는다 -> A도 {reach}번 시도하면 얻던 정답률을 B는 1번에 얻는다")
else:
    print(f"A 곡선은 k={max(K_LIST)}까지 이 높이에 닿지 않는다. K_LIST를 더 늘려 보자")

## 3. 곡선 그리기

두 곡선과 **B의 pass@1 수평선**을 한 그림에 그린다. 강의 노트의 곡선 그림처럼, 수평선이 A 곡선의 어디에 걸리는지가 판정의 근거다. k가 1부터 200까지 넓게 퍼지므로 가로축은 로그 눈금(1, 10, 100)으로 그린다. 그림 속 글자는 Colab에 한글 글꼴이 없어도 깨지지 않도록 영어로 쓴다.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4.5))
plt.plot(K_LIST, [curve_A[k] for k in K_LIST], "o-", color="#e03131", label="A: instruction only")
plt.plot(K_LIST, [curve_B[k] for k in K_LIST], "o-", color="#2f9e44", label="B: + 2 examples")
plt.axhline(b1, color="#2f9e44", linestyle="--", linewidth=1, label=f"B pass@1 = {b1:.2f}")
plt.xscale("log")
plt.xlabel("k (samples per question, log scale)")
plt.ylabel("pass@k")
plt.ylim(0, 1.05)
plt.title(f"pass@k on 24 Jeju tourism inquiries (n = {N_SAMPLES})")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### 3-1. 문항 단위로 보기 - B가 맞힌 문항은 A도 닿던 문항인가

비율만 비교하면 서로 다른 문항을 맞힌 경우를 놓친다. 그래서 문항 단위로 본다. A가 200번 중 한 번이라도 맞힌 문항(c > 0)의 집합을 **A의 커버리지**라 하자. B가 맞힌 문항이 그 안에 있는지 확인한다.

A가 200번 안에 한 번도 못 맞힌 문항이 있어도 바로 "B가 새 능력을 만들었다"고 읽으면 안 된다. 그 문항에서 A가 정답을 낼 확률이 0이 아니라 아주 작을 수도 있다. 아래 셀은 그 확률과, 한 번 나오기까지 평균 몇 번을 뽑아야 하는지(1/확률)도 함께 보여 준다.

In [ ]:
print(f"A가 {N_SAMPLES}번 안에 한 번이라도 맞힌 문항: {sum(c > 0 for c in counts_A)}/{len(eval_set)}")
print(f"B가 {N_SAMPLES}번 안에 한 번이라도 맞힌 문항: {sum(c > 0 for c in counts_B)}/{len(eval_set)}")
print()
print(f"B는 맞혔는데 A는 {N_SAMPLES}번 안에 한 번도 못 맞힌 문항:")
for (sentence, gold), c_a, c_b in zip(eval_set, counts_A, counts_B):
    if c_b > 0 and c_a == 0:
        prefix_ids = tokenizer(PROMPT_A.format(query=sentence), return_tensors="pt").input_ids
        with torch.no_grad():
            probs = torch.softmax(model(prefix_ids).logits[0, -1], dim=-1)
        p_gold = probs[tokenizer(" " + gold, add_special_tokens=False).input_ids[0]].item()
        print(f"  {sentence}  A의 정답 확률 {p_gold:.4f} -> 한 번 나오려면 평균 약 {1 / p_gold:.0f}번")

### 3-2. judge 판정을 여러 번 뽑아 다수결하면 false acceptance가 줄어드나

judge 판정도 pass@k처럼 여러 번 뽑을 수 있다. 판정을 k번 뽑아 **다수결**로 정하면 오답 승인이 줄어들까. 다수결은 pass@k와 다른 규칙이라는 점에 주의하자. pass@k는 "한 번이라도 맞으면 통과"이고, 다수결은 "절반 넘게 나온 쪽"이다.

1-6의 틀린 문항마다 "판정이 정답으로 나올 확률"을 세 방식으로 비교한다. 뽑기 결과가 흔들리지 않도록 시뮬레이션 대신 확률을 직접 계산한다.

- **판정 1번 뽑기**: P(정답) 그대로
- **판정 5번 뽑아 다수결**: 이항분포로 계산한 "정답이 과반일 확률"
- **확률이 높은 쪽 고르기** (1-6의 방식): 다수결을 무한히 한 것과 같다. P(정답)이 0.5보다 크면 언제나 정답

실행 전에 예상해 보자. P(정답)이 0.5보다 큰 문항에서 뽑는 횟수를 늘리면 "정답"이 과반일 확률은 커질까, 작아질까. (관찰용이다. 운영에서는 판정을 k번 뽑는 비용이 든다)

In [ ]:
J_K = 5  # 판정을 뽑는 횟수 (동점이 없도록 홀수)

def majority_prob(p, k=J_K):
    # 판정을 k번 뽑았을 때 "정답"이 과반일 확률 (이항분포)
    return sum(math.comb(k, i) * p**i * (1 - p)**(k - i) for i in range(k // 2 + 1, k + 1))

wrong = [(s, pc) for (s, g, p), pc in zip(rule_results, judge_nogold) if g != p]
print(f"{'1번 뽑기':>8} {str(J_K) + '번 다수결':>10} {'확률 높은 쪽':>10}   문의")
for s, pc in wrong:
    print(f"{pc:>10.2f} {majority_prob(pc):>12.2f} {float(pc > 0.5):>12.2f}   {s}")

print()
print(f"틀린 문항 {len(wrong)}건에서 기대되는 false acceptance 건수")
print(f"  판정 1번 뽑기:            {sum(pc for _, pc in wrong):.2f}건")
print(f"  판정 {J_K}번 뽑아 다수결:     {sum(majority_prob(pc) for _, pc in wrong):.2f}건")
print(f"  확률이 높은 쪽 (1-6 방식): {sum(pc > 0.5 for _, pc in wrong)}건")

## 4. 확인 질문

1. 규칙 채점의 정확도와 부류별 재현율은? 어느 부류가 어느 부류로 새나요?
2. 정답지 없는 judge의 false acceptance는 몇 건이었나요? 정답지를 넣었을 때 판정 단어는 바뀌었나요? 맞은 문항과 틀린 문항의 P(정답) 범위는 어떻게 달라졌나요?
3. A 곡선은 몇 번째 k에서 B의 pass@1 높이에 닿았나요? A 곡선의 끝은 평평해졌나요?
4. B의 pass@1 상승은 새 능력인가요, 탐색 효율 개선인가요? 곡선과 3-1의 문항 단위 결과를 근거로 한 문장으로 판정해 보세요.
5. 판정을 1번 뽑을 때보다 5번 뽑아 다수결할 때 false acceptance가 줄었나요, 늘었나요? 다수결이 judge의 치우침을 고치지 못하는 이유를 한 문장으로 써 보세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-05/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.